In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json, pickle
import math
import librosa
import numpy as np
from sys import getsizeof

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/NavaDataset/Data" # Path to dataset
PICKLE_PATH = "/content/drive/MyDrive/AUT/Project/Features/DATA_MFCC30_MEL128.pkl" # Path to pickle file used to save features

In [ ]:
SAMPLE_RATE = 22050
N_MFCC = 30 # Number of MFCC coefficients to extract
N_MELS = 128 # Number of Mel bands to generate
N_FFT = 4096 # Interval we consider to apply FFT. Measured in # of samples
HOP_LENGTH = 2048 # Sliding window for FFT. Measured in # of samples
SEGMENT_DURATION = 20 # SECONDS
SAMPLES_PER_SEGMENT = SAMPLE_RATE * SEGMENT_DURATION
EXPECTED_NUM_VECTORS_PER_SEGMENT = math.ceil(SAMPLES_PER_SEGMENT / HOP_LENGTH)

In [ ]:
class Track:
    """ Each track object is an audio file in the dataset """

    def __init__(self, filename):
        self.filename = filename.replace(".mp3", "").strip()
        self.path = os.path.join(DATASET_PATH, filename)
        self.signal, self.sr = librosa.load(path=str(self.path), sr= SAMPLE_RATE)

    def get_number_of_segments(self): # Number of segments we want to divide sample tracks into
        """ Calculates number of segments in an audiofile, considering each segment is 20 seconds """
        track_duration = librosa.get_duration(y=self.signal, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH)
        num_samples = track_duration * self.sr # Total number of segments in an audiofile
        num_segments = int(num_samples / SAMPLES_PER_SEGMENT)
        return num_segments

    def get_labels(self):
        """ Returns label of instrument and scale of the audio file"""
        instrument_label = self.filename[0]
        scale_label = self.filename[2]
        return instrument_label, scale_label

In [ ]:
class Segment:
    """ Each segment is part of a track, with duration of 20 seconds """

    def __init__(self, track, start, finish):
        self.signal = track.signal[start:finish]
        self.sample_rate = track.sr

    def extract_mfccs(self):
        """ Calculates MFCC, ΔMFCC and ΔΔMFCC and returns concatination of them"""
        mfcc = librosa.feature.mfcc(y=self.signal, sr=self.sample_rate, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
        delta_mfcc = librosa.feature.delta(data=mfcc)
        delta2_mfcc = librosa.feature.delta(data=mfcc, order=2)
        comprehensive_mfccs = np.concatenate((mfcc, delta_mfcc, delta2_mfcc), dtype=np.float32)
        return comprehensive_mfccs.T

    def extract_mel_spectrogram(self):
        """ Calculates MEL Spectrogram on a logarithmic basis"""
        mel_spectrogram = librosa.feature.melspectrogram(y=self.signal, sr=self.sample_rate, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
        log_mel_spectrogram = librosa.power_to_db(mel_spectrogram)
        return log_mel_spectrogram.T

In [ ]:
def save_data():
    """ Saves features to json/pickle file """
    with open(PICKLE_PATH, "wb") as output:
        pickle.dump(DATA, output)

In [ ]:
def store_features():
    """
    Extracts features from music dataset and saves them into a json file along witgh genre labels
    """

    for _, _, files in os.walk(DATASET_PATH):

        # loop through all musics in dataset folder
        for i, file in enumerate(files):

            # Process audio files
            new_track = Track(file)
            print("============================================")
            print(f"Processing track {i+1}/{len(files)}: {file}")

            # Process all segments of audio file
            for s in range (new_track.get_number_of_segments()):

                # Calculate start and finish sample for current segment
                start_sample = SAMPLES_PER_SEGMENT * s
                finish_sample = start_sample + SAMPLES_PER_SEGMENT
                new_segment = Segment(new_track, start_sample, finish_sample)

                # Extract MFCCs and Mel-Spectrogram from current segment, frame by frame
                comprehensive_mfccs = new_segment.extract_mfccs()
                # mel_spectrogram = new_segment.extract_mel_spectrogram()

                # print(comprehensive_mfccs.shape)
                # print(mel_spectrogram.shape)

                # Store only features with expected number of vectors
                if len(comprehensive_mfccs) == len(mel_spectrogram) == EXPECTED_NUM_VECTORS_PER_SEGMENT:
                # if len(comprehensive_mfccs) == EXPECTED_NUM_VECTORS_PER_SEGMENT:
                    DATA["mfccs"].append(comprehensive_mfccs.tolist())
                    DATA["mel_spec"].append(mel_spectrogram.tolist())
                    instrument_label, scale_label = new_track.get_labels()
                    DATA["labels"]["instrument"].append(instrument_label)
                    DATA["labels"]["scale"].append(scale_label)
                    # print(f"Storing track: {new_track.filename}, segment: {s+1}")
                del new_segment
            #     break
            # break
            del new_track
    save_data()

In [ ]:
store_features()

Streaming output truncated to the last 5000 lines.
test
Processing track 115/1785: 2_6_23_15.mp3
train
Processing track 116/1785: 1_5_32_00.mp3
train
Processing track 117/1785: 3_4_17_23.mp3
train
Processing track 118/1785: 2_1_15_06.mp3
train
Processing track 119/1785: 2_6_15_01.mp3
train
Processing track 120/1785: 3_4_17_09.mp3
train
Processing track 121/1785: 1_5_28_01.mp3
test
Processing track 122/1785: 0_3_03_17.mp3.mp3
train
Processing track 123/1785: 2_5_23_05.mp3
train
Processing track 124/1785: 2_1_23_02.mp3
train
Processing track 125/1785: 0_1_22_06.mp3
train
Processing track 126/1785: 2_1_23_12.mp3
train
Processing track 127/1785: 3_1_17_02.mp3
dev
Processing track 128/1785: 2_2_15_04.mp3
dev
Processing track 129/1785: 2_3_01_02.mp3
train
Processing track 130/1785: 2_6_15_13.mp3
train
Processing track 131/1785: 3_1_17_07.mp3
dev
Processing track 132/1785: 2_0_21_05.mp3
train
Processing track 133/1785: 4_2_25_18.mp3
train
Processing track 134/1785: 2_1_23_15.mp3
train
Process

In [ ]:
print(np.array(DATA["mfccs"]).shape)
print(np.array(DATA["mel_spec"]).shape)

(9001, 216, 90)
(9001, 216, 128)
